# Automatic Speech Recognition (ASR)

Welcome to this introductory guide to Automatic Speech Recognition (ASR)! ASR is a fascinating field within Artificial Intelligence (AI) that focuses on converting spoken language into written text. Think about voice assistants like Siri or Alexa, dictation software, or automatic captioning on videos – these are all powered by ASR technology.

**What is ASR?**

At its core, ASR systems take an audio waveform (your voice) as input and produce a sequence of words (text) as output. This process involves several complex steps, including:

1.  **Signal Processing:** Cleaning the audio signal, removing noise, and extracting relevant features.
2.  **Acoustic Modeling:** Mapping the audio features to basic units of sound, like phonemes /a/, /o/ ...
3.  **Language Modeling:** Understanding the probability of sequences of words occurring in a given language. This helps the system choose the most likely words.
4.  **Decoding:** Combining the acoustic and language models to find the most probable sequence of words corresponding to the input audio.

Modern ASR heavily relies on deep learning techniques, which have significantly improved accuracy over the past decade.

## Key ASR Models: Wav2Vec 2.0 and Whisper

Two prominent models have significantly advanced the field of ASR: Wav2Vec 2.0 and Whisper. These models leverage large amounts of data and sophisticated deep learning architectures to achieve state-of-the-art performance.

**Wav2Vec 2.0 (from Meta AI):** This model uses a clever approach called self-supervised learning. Instead of needing vast amounts of transcribed audio (audio paired with text), Wav2Vec 2.0 learns powerful representations directly from raw audio data. It masks parts of the audio input and tries to predict them based on the surrounding context, similar to how language models like BERT work with text. Once pre-trained on unlabeled audio, Wav2Vec 2.0 can be fine-tuned with a relatively small amount of labeled data for specific ASR tasks and languages, making it very versatile.

**Whisper (from OpenAI):** Whisper takes a different approach. It's trained on a massive and diverse dataset comprising 680,000 hours of multilingual and multitask supervised data collected from the web. This extensive training allows Whisper to perform remarkably well across a wide range of languages, accents, and noisy conditions, often without needing specific fine-tuning (a capability known as zero-shot performance). It's designed as an end-to-end system, directly mapping audio to text.

## Practical Examples: Arabic and Moroccan Darija ASR

Let's see how we can use pre-trained models for ASR tasks, specifically focusing on Arabic and Moroccan Darija. We will use models available on the Hugging Face Hub, a platform hosting thousands of pre-trained models.

First, we need to install the necessary libraries. If you haven't already, run the following cell. Note: Installation might take a few minutes, and Whisper might require `ffmpeg` to be installed on your system (`sudo apt update && sudo apt install ffmpeg`).

P.S: You don't necessarily need a GPU for this notebook.

In [1]:
!pip install transformers torch soundfile librosa speechbrain
!sudo apt update && sudo apt install ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 43.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 34.6 MB/s eta 0:00:00
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]    
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease3m 
Get:10 http://archive.ubuntu.com/u

In [2]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 14.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


### Example 1: Arabic ASR with Wav2Vec 2.0

We will use a Wav2Vec 2.0 model fine-tuned for Arabic. The `transformers` library from Hugging Face makes it easy to load and use these models. We'll need an audio file in Arabic to test this. For demonstration purposes, we'll load a sample from the `datasets` library, but you can replace `\'common_voice\' ` and the specific sample index with your own audio file path after loading it appropriately (e.g., using `librosa` or `soundfile`). Remember that the audio needs to be sampled at 16kHz for most Wav2Vec models.

In [3]:
import torch
import librosa
from datasets import load_dataset, Audio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

# Load a pre-trained Arabic ASR model and processor
model_name_wav2vec = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"
processor_wav2vec = Wav2Vec2Processor.from_pretrained(model_name_wav2vec) # Processor for Audio
model_wav2vec = Wav2Vec2ForCTC.from_pretrained(model_name_wav2vec)

# Load a sample Arabic audio file (e.g., from Common Voice dataset)
arabic_audio_sample = None
original_sentence = "(Could not load sample)"

# Load a small part of the dataset for demonstration
common_voice_ar = load_dataset("ayoubkirouane/Arabic_common_voice_11_0", split="train[:1%]")
# Resample the audio to 16kHz as required by the model
common_voice_ar = common_voice_ar.cast_column("audio", Audio(sampling_rate=16000))
# Select the first audio sample
arabic_audio_sample = common_voice_ar[0]["audio"]["array"]
sampling_rate = common_voice_ar[0]["audio"]["sampling_rate"]
original_sentence = common_voice_ar[0]['sentence']
print(f"Loaded sample audio with rate: {sampling_rate} Hz")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

...

data/train-00000-of-00001-275ca77ae31f2d(…):   0%|          | 0.00/281M [00:00<?, ?B/s]

data/test-00000-of-00001-9b36e8c5c0871fe(…):   0%|          | 0.00/297M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10438 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10440 [00:00<?, ? examples/s]

Loaded sample audio with rate: 16000 Hz


In [4]:
# Preprocess the audio
input_values = processor_wav2vec(arabic_audio_sample, sampling_rate=sampling_rate, return_tensors="pt").input_values

In [5]:
# Perform inference
with torch.no_grad():
    logits = model_wav2vec(input_values).logits

In [6]:
print(logits)

tensor([[[ 15.7974, -18.9776, -18.6947,  ...,  -6.8068,  -6.2553,  -6.4282],
         [ 15.7586, -19.1368, -18.8573,  ...,  -6.8094,  -6.1801,  -6.4290],
         [ 15.8280, -19.3175, -19.0202,  ...,  -6.8569,  -6.1166,  -6.4215],
         ...,
         [ 16.2255, -19.4670, -19.2203,  ...,  -6.7365,  -4.7465,  -5.8244],
         [ 15.8793, -19.2215, -18.9713,  ...,  -6.4134,  -4.3094,  -6.0376],
         [  2.3515,  -7.5080,  -7.3994,  ...,  -1.6064,  -1.3047,  -0.7052]]])


In [7]:
# Decode the predicted IDs
predicted_ids = torch.argmax(logits, dim=-1)
transcription = processor_wav2vec.batch_decode(predicted_ids)[0]

In [8]:
print(f"Original Sentence (from dataset): {original_sentence}")
print(f"Wav2Vec Transcription: {transcription}")

Original Sentence (from dataset): عمي هو أخو أبي.
Wav2Vec Transcription: عمي هو أخ أبي


### Example 2: Multilingual ASR with Whisper

Whisper models are known for their strong multilingual capabilities out-of-the-box. We can easily use them via the `transformers` pipeline. This pipeline handles the pre-processing, model inference, and post-processing for us. We can test it on the same Arabic audio sample loaded earlier, or you can provide a path to any audio file (Whisper handles various formats if `ffmpeg` is installed). Whisper automatically detects the language, but you can also specify it for potentially better results.

In [9]:
from transformers import pipeline
import numpy as np

# Load the ASR pipeline with a Whisper model
whisper_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-base")
print("Whisper pipeline loaded successfully.")

# Use the Arabic audio sample loaded in Example 1 if available
if arabic_audio_sample is not None:
    print("Transcribing Arabic sample with Whisper...")

    # 1. Ensure the audio is a 1D float32 numpy array
    audio_array = np.array(arabic_audio_sample, dtype=np.float32)
    if audio_array.ndim > 1:
        audio_array = np.squeeze(audio_array) # Flatten to 1D if it's nested
        audio_array = librosa.to_mono(audio_array) # Convert to mono if it's stereo


    # 2. Package it into a dictionary with the required sampling rate (16kHz)
    # Note: If your original audio is NOT 16kHz, you will need to resample it first
    # (e.g., using librosa.resample) before passing it here.
    audio_input_whisper = {
        "array": audio_array,
        "sampling_rate": 16000
    }

    # Perform transcription
    # result = whisper_pipeline(audio_input_whisper)
    result = whisper_pipeline(audio_input_whisper, chunk_length_s=30)
    transcription_whisper = result["text"]

    print(f"Original Sentence (from dataset): {original_sentence}")
    print(f"Whisper Transcription: {transcription_whisper}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Whisper pipeline loaded successfully.
Transcribing Arabic sample with Whisper...


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


Original Sentence (from dataset): عمي هو أخو أبي.
Whisper Transcription:  عمي حوة حوة بي


### Example 3: Moroccan Darija ASR with SpeechBrain

For Moroccan Darija, we can use a model specifically trained for it, available through the `speechbrain` library, which also integrates with Hugging Face. This example demonstrates how to load the `speechbrain/asr-wav2vec2-dvoice-darija` model and use it for transcription. Similar to the previous example, you'll need a Darija audio file (sampled at 16kHz). We'll use a placeholder here; you should replace `'/content/path/to/your/darija_audio.wav' ` with the actual path to your audio file.

Challenge the model by intentionally creating a sample that lead to poor transcription results. Think about factors that could make transcription difficult.

In [10]:
from speechbrain.pretrained import EncoderASR
import torchaudio
import os

# Load the pre-trained Darija ASR model
# This will download the model from Hugging Face Hub if not already cached
asr_model_sb = EncoderASR.from_hparams(source="speechbrain/asr-wav2vec2-dvoice-darija", savedir="pretrained_models/asr-wav2vec2-dvoice-darija")
print("Darija ASR model loaded successfully.")

/tmp/ipykernel_5982/3193684589.py:1: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import EncoderASR
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/asr-wav2vec2-dvoice-darija' if not cached


hyperparams.yaml: 0.00B [00:00, ?B/s]

/usr/lib/python3.12/importlib/__init__.py:90: UserWarning: Module 'speechbrain.lobes.models.huggingface_transformers' was deprecated, redirecting to 'speechbrain.integrations.huggingface'. Please update your script.
  return _bootstrap._gcd_import(name[level:], package, level)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

...

asr.ckpt:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch tokenizer.ckpt: Fetching from HuggingFace Hub 'speechbrain/asr-wav2vec2-dvoice-darija' if not cached


tokenizer.ckpt:   0%|          | 0.00/238k [00:00<?, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: wav2vec2, asr, tokenizer


Darija ASR model loaded successfully.


In [ ]:
import soundfile as sf
fichier_original = "/content/jatk-el-khedma_aby16M04.wav"   
y, sr = librosa.load(fichier_original, sr=16000)
print(f"Audio chargé : durée = {len(y)/sr:.2f} sec, sr={sr} Hz")

/tmp/ipykernel_5982/378845039.py:4: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(fichier_original, sr=16000)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


FileNotFoundError: [Errno 2] No such file or directory: '/content/jatk-el-khedma_aby16M04.wav'

In [ ]:
# 1. Bruit blanc (SNR faible)
bruit = np.random.normal(0, 0.1, y.shape)
y_bruit = y + bruit
sf.write("/content/bruit_blanc.wav", y_bruit, sr)

# 2. Accélération (parole plus rapide)
y_accelere = librosa.effects.time_stretch(y, rate=1.4)   # 40% plus rapide
sf.write("/content/accelere.wav", y_accelere, sr)

# 3. Réverbération simple (echo)
echo = np.zeros_like(y)
echo[2000:] = y[:-2000] * 0.5   # décalage + atténuation
y_reverb = y + echo
sf.write("/content/reverb.wav", y_reverb, sr)

print("3 fichiers dégradés créés :")
print(" - /content/bruit_blanc.wav")
print(" - /content/accelere.wav")
print(" - /content/reverb.wav")

3 fichiers dégradés créés :
 - /content/bruit_blanc.wav
 - /content/accelere.wav
 - /content/reverb.wav


In [ ]:
print("\n=== Transcription sur audio original ===")
print(asr_model_sb.transcribe_file(fichier_original))

print("\n=== Transcription avec bruit blanc ===")
print(asr_model_sb.transcribe_file("/content/bruit_blanc.wav"))

print("\n=== Transcription sur parole accélérée ===")
print(asr_model_sb.transcribe_file("/content/accelere.wav"))

print("\n=== Transcription avec réverbération ===")
print(asr_model_sb.transcribe_file("/content/reverb.wav"))


=== Transcription sur audio original ===
شوفكيه  الخدمهما عندي منقولزوي بزاف

=== Transcription avec bruit blanc ===
تحهعه

=== Transcription sur parole accélérée ===
شفيه جشلهمعتموزع

=== Transcription avec réverbération ===
شوفكي دنامعد مارورودزي


In [ ]:
# Affiche les différences pour voir où le modèle échoue
original_text = asr_model_sb.transcribe_file(fichier_original)
noisy_text = asr_model_sb.transcribe_file("/content/bruit_blanc.wav")

print("\n Texte original :", original_text)
print(" Texte avec bruit :", noisy_text)
print(" Le modèle a mal transcrit à cause du bruit de fond qui masque les phonèmes.")


✅ Texte original : شوفكيه  الخدمهما عندي منقولزوي بزاف
❌ Texte avec bruit : تحهعه
👉 Le modèle a mal transcrit à cause du bruit de fond qui masque les phonèmes.


## Finding and Using Moroccan Darija Datasets

While pre-trained models are convenient, you may still want to fine-tune a model on your own data. For Moroccan Darija, access to the right datasets is essential, and the Hugging Face Hub is a good place to start.

The `speechbrain/asr-wav2vec2-dvoice-darija` model used in this notebook was trained on the **DVoice Darija** dataset, which is available [on Zenodo](https://zenodo.org/records/6342622) and may also appear in Hugging Face mirrors.

Two datasets are commonly used for Darija:

* **DVoice** - available on Zenodo.
* **DODA** - integrates with the Hugging Face `datasets` library and is often used for fine-tuning Wav2Vec2 models.

If your goal is inference only, several pre-trained Darija models are available:

* [`speechbrain/asr-wav2vec2-dvoice-darija`](https://huggingface.co/speechbrain/asr-wav2vec2-dvoice-darija)
* [`boumehdi/wav2vec2-large-xlsr-moroccan-darija`](https://huggingface.co/boumehdi/wav2vec2-large-xlsr-moroccan-darija)
* [`KandirResearch/Whisper-Small-Darija`](https://huggingface.co/KandirResearch/Whisper-Small-Darija)
* [`atlasia/moulsot.v0.3`](https://huggingface.co/atlasia/moulsot.v0.3)

For additional guidance or questions related to your project, feel free to contact **Yassine El Kheir**.

## Understanding Check

1. Outline the major steps required to run inference on an existing ASR model.
2. Identify two potential challenges specific to Darija ASR.

### Suggested answers

**1. Major steps to run inference on an existing ASR model**

1. Load the pre-trained model and its tokenizer or processor.
   - Example: `Wav2Vec2ForCTC.from_pretrained()`, `EncoderASR.from_hparams()`, or `pipeline("automatic-speech-recognition")`.
2. Load and prepare the audio file.
   - Read the audio with `librosa.load()` or `soundfile.read()`.
   - Resample it to the required sampling rate, usually 16 kHz for Wav2Vec2 and Whisper.
   - Convert stereo audio to mono if needed.
3. Preprocess the audio into the model’s expected input format.
   - For Wav2Vec2: use the processor to extract `input_values`.
   - For Whisper: pass an audio dictionary such as `{"array": audio, "sampling_rate": 16000}`.
4. Run inference.
   - Use `torch.no_grad()` to avoid computing gradients.
   - The model outputs logits for each time step.
5. Decode the logits into text.
   - Take `argmax` over the token dimension to get token IDs.
   - Use the tokenizer or processor to convert IDs into readable text.
   - For CTC models, repeated and blank tokens are handled during decoding.
6. Post-process and display the transcription.
   - Optionally clean punctuation or casing.
   - Print or return the final result.

**2. Two potential challenges specific to Darija ASR**

1. Lack of a standardized writing system.
   - Darija is mainly a spoken dialect with no single official orthography.
   - The same word can be written in multiple ways, which increases transcription variance.
2. Code-switching and multilingual mixing.
   - Darija speakers often mix Darija, Modern Standard Arabic, French, and Amazigh in the same sentence.
   - This can reduce recognition quality for mixed or out-of-vocabulary words.